# Week 2 Lab: Dynamic Programming
**MSDS684 - Reinforcement Learning**

Implement policy iteration and value iteration to solve GridWorld.

## Conda Environment Setup (Miniconda)

Run the following commands in your terminal to create and activate a conda environment for this lab:

```bash
# Create a new conda environment
conda create -n rl-lab2 python=3.11 -y

# Activate the environment
conda activate rl-lab2

# Install required packages
conda install numpy matplotlib -y
pip install gymnasium

# Register the environment as a Jupyter kernel
pip install ipykernel
python -m ipykernel install --user --name rl-lab2 --display-name "Python (rl-lab2)"
```

After setup, select the **"Python (rl-lab2)"** kernel in this notebook.

## Setup & Imports
- Import NumPy, Matplotlib, Gymnasium
- Set random seeds for reproducibility

In [1]:
%pip install -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import gymnasium as gym
import time
from typing import Optional

np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## Custom GridWorld Environment
Build a custom GridWorld (4x4 or larger) following Gymnasium's API.

### Requirements:
- Configurable rewards, obstacles, and terminal states
- Support for **deterministic** transitions
- Support for **stochastic** transitions (e.g., 80% intended direction, 10% each perpendicular)
- Expose full transition model `P(s', r | s, a)` for DP access
- Implement `env.unwrapped` style access to transition dynamics

In [ ]:
class GridWorldEnv(gym.Env):
    """
    Custom GridWorld environment following Gymnasium's API.
    Exposes full transition model P[s][a] for dynamic programming.
    """

    metadata = {"render_modes": ["human"]}

    # Actions: 0=up, 1=right, 2=down, 3=left
    ACTION_UP = 0
    ACTION_RIGHT = 1
    ACTION_DOWN = 2
    ACTION_LEFT = 3

    # Direction vectors (row_delta, col_delta) for each action
    ACTION_DELTAS = {
        0: (-1, 0),  # up
        1: (0, 1),   # right
        2: (1, 0),   # down
        3: (0, -1),  # left
    }

    # Perpendicular actions for stochastic transitions
    PERPENDICULAR = {
        0: [3, 1],  # up -> [left, right]
        1: [0, 2],  # right -> [up, down]
        2: [1, 3],  # down -> [right, left]
        3: [2, 0],  # left -> [down, up]
    }

    def __init__(
        self,
        grid_size: tuple = (4, 4),
        start: tuple = (0, 0),
        goals: Optional[list] = None,
        obstacles: Optional[list] = None,
        step_reward: float = -1.0,
        goal_reward: float = 0.0,
        obstacle_reward: float = -10.0,
        slip_prob: float = 0.0,
        gamma: float = 1.0,
        render_mode: Optional[str] = None,
    ):
        super().__init__()

        self.nrow, self.ncol = grid_size
        self.n_states = self.nrow * self.ncol
        self.start = start
        self.goals = goals if goals is not None else [(self.nrow - 1, self.ncol - 1)]
        self.obstacles = obstacles if obstacles is not None else []
        self.step_reward = step_reward
        self.goal_reward = goal_reward
        self.obstacle_reward = obstacle_reward
        self.slip_prob = slip_prob  # 0.0 = deterministic, e.g. 0.1 for 80/10/10
        self.gamma = gamma
        self.render_mode = render_mode

        self.observation_space = gym.spaces.Discrete(self.n_states)
        self.action_space = gym.spaces.Discrete(4)

        # Build the full transition model
        self.P = self._build_transition_model()

    def _pos_to_state(self, row, col):
        return row * self.ncol + col

    def _state_to_pos(self, state):
        return divmod(state, self.ncol)

    def _is_valid(self, row, col):
        return 0 <= row < self.nrow and 0 <= col < self.ncol

    def _get_next_state_and_reward(self, row, col, action):
        """Given a position and action, return (next_state, reward, terminated)."""
        # Terminal states are absorbing — no transitions out
        if (row, col) in self.goals:
            s = self._pos_to_state(row, col)
            return s, 0.0, True

        dr, dc = self.ACTION_DELTAS[action]
        new_row, new_col = row + dr, col + dc

        # Boundary check: stay in place if invalid or obstacle
        if not self._is_valid(new_row, new_col) or (new_row, new_col) in self.obstacles:
            new_row, new_col = row, col

        next_state = self._pos_to_state(new_row, new_col)
        terminated = (new_row, new_col) in self.goals

        if terminated:
            reward = self.goal_reward
        elif (new_row, new_col) in self.obstacles:
            reward = self.obstacle_reward
        else:
            reward = self.step_reward

        return next_state, reward, terminated

    def _build_transition_model(self):
        """Build P[s][a] = [(prob, next_state, reward, terminated), ...]"""
        P = {}
        for s in range(self.n_states):
            P[s] = {}
            row, col = self._state_to_pos(s)

            for a in range(4):
                transitions = []

                if self.slip_prob == 0.0:
                    # Deterministic
                    ns, r, done = self._get_next_state_and_reward(row, col, a)
                    transitions.append((1.0, ns, r, done))
                else:
                    # Stochastic: intended direction + perpendicular slips
                    intended_prob = 1.0 - 2 * self.slip_prob

                    # Intended direction
                    ns, r, done = self._get_next_state_and_reward(row, col, a)
                    transitions.append((intended_prob, ns, r, done))

                    # Perpendicular directions
                    for perp_action in self.PERPENDICULAR[a]:
                        ns, r, done = self._get_next_state_and_reward(row, col, perp_action)
                        transitions.append((self.slip_prob, ns, r, done))

                P[s][a] = transitions

        return P

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_state = self._pos_to_state(*self.start)
        return self.current_state, {}

    def step(self, action):
        row, col = self._state_to_pos(self.current_state)

        if self.slip_prob == 0.0:
            actual_action = action
        else:
            probs = [1.0 - 2 * self.slip_prob, self.slip_prob, self.slip_prob]
            actions = [action] + self.PERPENDICULAR[action]
            actual_action = self.np_random.choice(actions, p=probs)

        next_state, reward, terminated = self._get_next_state_and_reward(row, col, actual_action)
        self.current_state = next_state

        return next_state, reward, terminated, False, {}

    def render(self):
        row, col = self._state_to_pos(self.current_state)
        grid = np.full((self.nrow, self.ncol), ".", dtype=object)
        for r, c in self.obstacles:
            grid[r, c] = "X"
        for r, c in self.goals:
            grid[r, c] = "G"
        grid[row, col] = "A"
        print("\n".join(" ".join(r) for r in grid))
        print()

## Visualization Helpers
Create reusable plotting functions before implementing algorithms.

### Functions needed:
- **Value function heatmap** — display V(s) as a colored grid using Matplotlib `imshow`
- **Policy arrow plot** — display π(s) as arrows using `quiver`
- **Convergence curves** — plot iterations vs. delta and wall-clock time

## Policy Evaluation (Prediction Problem)
Compute V(s) for a given policy π.

### Synchronous Policy Evaluation
- Use a separate array for V_new; update all states before copying back
- Sweep over all states: $V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma V_k(s')]$
- Track delta (max change) per iteration for convergence check

### In-Place Policy Evaluation
- Update V(s) in-place as we sweep (no separate V_new array)
- Typically converges faster since updates use latest values immediately

## Policy Improvement
Given V(s), produce a greedy policy.

- For each state, compute Q(s,a) for all actions
- $\pi'(s) = \arg\max_a \sum_{s',r} p(s',r|s,a)[r + \gamma V(s')]$
- Track whether policy changed (policy_stable flag)

## Policy Iteration
Alternate between evaluation and improvement until policy is stable.

### Synchronous Policy Iteration
- Initialize policy arbitrarily
- **Evaluate**: run synchronous policy evaluation to convergence
- **Improve**: derive greedy policy from V(s)
- If policy unchanged → done; else repeat
- Record V(s) and π(s) at each outer iteration for visualization

### In-Place Policy Iteration
- Same loop, but use in-place policy evaluation in the inner loop

## Value Iteration
Combine truncated evaluation (1 sweep) with improvement in a single update.

### Synchronous Value Iteration
- $V_{k+1}(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma V_k(s')]$
- Use separate V_new array per sweep
- Converge when delta < θ, then extract greedy policy

### In-Place Value Iteration
- Same as above but update V(s) in-place during sweep

## Experiment: Deterministic GridWorld
Test all four algorithm variants on a deterministic GridWorld.

### Steps:
- Configure GridWorld with 100% intended-direction transitions
- Run synchronous & in-place policy iteration
- Run synchronous & in-place value iteration
- **Visualize**: heatmaps of V(s) at each iteration, policy arrows, convergence curves
- Record iteration counts and wall-clock times

## Experiment: Stochastic GridWorld
Test all four algorithm variants on a stochastic GridWorld.

### Steps:
- Configure GridWorld with 80% intended, 10% each perpendicular
- Run all four algorithm variants
- **Visualize**: heatmaps, policy arrows, convergence curves
- Compare results to deterministic case

## Gymnasium FrozenLake-v1
Apply DP implementations to a standard Gymnasium environment.

### Steps:
- Load `FrozenLake-v1` (4x4 and/or 8x8)
- Access transition dynamics via `env.unwrapped.P`
- Run policy iteration and value iteration
- Visualize results and compare convergence

## Convergence Comparison & Analysis
Document which algorithm converges faster and why.

### Compare across all experiments:
- Number of iterations to convergence
- Wall-clock time
- Synchronous vs. in-place performance
- Policy iteration vs. value iteration
- Deterministic vs. stochastic environments

### Discussion points:
- Why does in-place typically converge faster?
- When is policy iteration preferred over value iteration (and vice versa)?
- How does stochasticity affect convergence?
- Generalized Policy Iteration (GPI) framework — how do these algorithms fit?